# Backprop, Jacobians, Hessians: Why First-Order Wins

Companion notebook for the blog post [Backprop, Jacobians, Hessians: Why First-Order Wins](https://sesen.ai/blog/backprop-jacobians-hessians-first-order-wins).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/backprop_jacobian_hessian.ipynb)

We fit a 2-layer GELU network with 25 parameters to 40 noisy samples of `sin(2πx)` three ways:
1. **SGD + momentum** — first-order, O(W) per step
2. **L-BFGS** — quasi-Newton, O(mW) per step with history m≈20
3. **Gauss-Newton (Levenberg-Marquardt)** — second-order via the outer-product Hessian approximation from Bishop §5.4.2

And inspect: the Jacobian ∂y/∂x, the Hessian eigenvalue spectrum (which has negative eigenvalues on this non-convex loss), and why the activation function's second derivative matters.


## Setup

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.animation import FuncAnimation, PillowWriter

torch.manual_seed(0)
np.random.seed(0)


## Data: 40 noisy samples of sin(2πx)

In [ ]:
def target(x):
    return torch.sin(2 * torch.pi * x)

N = 40
x_train = torch.linspace(0.05, 0.95, N, dtype=torch.float64).unsqueeze(1)
y_train = target(x_train) + 0.08 * torch.randn_like(x_train)
x_plot = torch.linspace(0, 1, 400, dtype=torch.float64).unsqueeze(1)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(x_plot, target(x_plot), 'k--', label='true sin(2πx)')
ax.scatter(x_train, y_train, s=18, color='#1B2D3D', label='observations')
ax.legend(); ax.grid(alpha=0.3); plt.show()


## The model: 2-layer GELU net (25 parameters)

We store parameters as a flat 25-vector. `predict` unpacks the vector into the four weight/bias tensors and runs the forward pass. We use GELU because Newton's method needs the activation's second derivative, which is well-defined for GELU but zero almost everywhere for ReLU.


In [ ]:
HIDDEN = 8
INIT_SEED = 7


def init_params(seed=INIT_SEED, hidden=HIDDEN):
    torch.manual_seed(seed)
    model = torch.nn.Sequential(
        torch.nn.Linear(1, hidden, dtype=torch.float64),
        torch.nn.GELU(),
        torch.nn.Linear(hidden, 1, dtype=torch.float64),
    )
    return torch.cat([p.detach().view(-1) for p in model.parameters()]).clone()


def unpack(vec, hidden=HIDDEN):
    o = 0
    W1 = vec[o:o + hidden].view(hidden, 1); o += hidden
    b1 = vec[o:o + hidden]; o += hidden
    W2 = vec[o:o + hidden].view(1, hidden); o += hidden
    b2 = vec[o:o + 1]
    return W1, b1, W2, b2


def predict(vec, x):
    W1, b1, W2, b2 = unpack(vec)
    return F.gelu(x @ W1.T + b1) @ W2.T + b2


def loss_from_vec(vec):
    return 0.5 * ((predict(vec, x_train) - y_train) ** 2).mean()


def residuals(vec):
    return (predict(vec, x_train) - y_train).squeeze(-1)


print('Parameter count W =', init_params().numel())


## Trainer 1: SGD with momentum and cosine-decay learning rate

In [ ]:
def train_sgd(n_iters=20000, lr=0.05, mom=0.95):
    theta = init_params().clone().requires_grad_(True)
    vel = torch.zeros_like(theta)
    hist = {'iter': [], 'wall': [], 'loss': [], 'params': []}
    t0 = time.perf_counter()
    for k in range(n_iters):
        lr_k = lr * (0.5 * (1 + np.cos(np.pi * k / n_iters))) + 1e-4
        loss = loss_from_vec(theta)
        grad = torch.autograd.grad(loss, theta)[0]
        with torch.no_grad():
            vel = mom * vel - lr_k * grad
            theta.add_(vel)
        if k % 100 == 0 or k == n_iters - 1:
            hist['iter'].append(k + 1)
            hist['wall'].append(time.perf_counter() - t0)
            hist['loss'].append(loss.item())
            hist['params'].append(theta.detach().clone())
    return theta.detach(), hist

sgd_theta, sgd_hist = train_sgd()
print(f'SGD final loss: {sgd_hist["loss"][-1]:.4e} in {sgd_hist["iter"][-1]} updates')


## Trainer 2: L-BFGS (quasi-Newton, approximates H⁻¹ from gradient history)

In [ ]:
def train_lbfgs(n_iters=40):
    theta = init_params().clone().requires_grad_(True)
    opt = torch.optim.LBFGS([theta], lr=1.0, max_iter=1, history_size=20,
                            line_search_fn='strong_wolfe',
                            tolerance_grad=1e-12, tolerance_change=1e-16)
    hist = {'iter': [], 'wall': [], 'loss': [], 'params': []}
    t0 = time.perf_counter()

    def closure():
        opt.zero_grad()
        loss = loss_from_vec(theta)
        loss.backward()
        return loss

    for k in range(n_iters):
        loss = opt.step(closure)
        hist['iter'].append(k + 1)
        hist['wall'].append(time.perf_counter() - t0)
        hist['loss'].append(loss.item())
        hist['params'].append(theta.detach().clone())
    return theta.detach(), hist

lbfgs_theta, lbfgs_hist = train_lbfgs()
print(f'L-BFGS final loss: {lbfgs_hist["loss"][-1]:.4e} in {lbfgs_hist["iter"][-1]} iters')


## Trainer 3: Gauss-Newton / Levenberg-Marquardt

Bishop §5.4.2: approximate the Hessian as `H ≈ J^T J` where `J` is the Jacobian of the residual vector with respect to parameters. By construction `J^T J` is positive semi-definite, so the Newton step is always a descent direction.


In [ ]:
def train_gauss_newton(n_iters=30, damping0=0.1):
    theta = init_params().clone().requires_grad_(True)
    damping = damping0
    hist = {'iter': [], 'wall': [], 'loss': [], 'params': []}
    t0 = time.perf_counter()
    I = torch.eye(theta.numel(), dtype=theta.dtype)
    for k in range(n_iters):
        J = torch.autograd.functional.jacobian(residuals, theta)  # (N, W)
        r = residuals(theta).detach()
        loss_val = 0.5 * (r @ r).item() / N
        JtJ = J.T @ J / N
        g = (J.T @ r) / N
        step = torch.linalg.solve(JtJ + damping * I, g)
        trial = theta.detach() - step
        trial_loss = 0.5 * (residuals(trial) @ residuals(trial)).item() / N
        if trial_loss < loss_val:
            theta = trial.clone().requires_grad_(True)
            damping = max(damping / 3, 1e-7)
        else:
            damping = min(damping * 3, 1e6)
        hist['iter'].append(k + 1)
        hist['wall'].append(time.perf_counter() - t0)
        hist['loss'].append(loss_from_vec(theta).item())
        hist['params'].append(theta.detach().clone())
    return theta.detach(), hist

gn_theta, gn_hist = train_gauss_newton()
print(f'Gauss-Newton final loss: {gn_hist["loss"][-1]:.4e} in {gn_hist["iter"][-1]} iters')


## Convergence per iteration and per wall-clock

Gauss-Newton wins iterations. L-BFGS wins wall-clock. SGD needs ~1000× more updates to reach the same loss, but each update is the cheapest.


In [ ]:
noise_floor = 0.5 * 0.08 ** 2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax in axes:
    ax.axhline(noise_floor, color='gray', ls='--', lw=0.8, label='noise floor')
    ax.set_yscale('log'); ax.grid(alpha=0.3, which='both')

ax = axes[0]
ax.semilogx(sgd_hist['iter'], sgd_hist['loss'], color='#3D9B8F', lw=1.8, label='SGD+mom')
ax.semilogx(lbfgs_hist['iter'], lbfgs_hist['loss'], color='#D4A24C', lw=2.0, marker='s', ms=3, label='L-BFGS')
ax.semilogx(gn_hist['iter'], gn_hist['loss'], color='#1B2D3D', lw=2.0, marker='o', ms=3, label='Gauss-Newton')
ax.set_xlabel('outer iteration'); ax.set_ylabel('training MSE'); ax.set_title('per iteration')
ax.legend()

ax = axes[1]
ax.loglog(sgd_hist['wall'], sgd_hist['loss'], color='#3D9B8F', lw=1.8, label='SGD+mom')
ax.loglog(lbfgs_hist['wall'], lbfgs_hist['loss'], color='#D4A24C', lw=2.0, marker='s', ms=3, label='L-BFGS')
ax.loglog(gn_hist['wall'], gn_hist['loss'], color='#1B2D3D', lw=2.0, marker='o', ms=3, label='Gauss-Newton')
ax.set_xlabel('wall-clock time (s)'); ax.set_ylabel('training MSE'); ax.set_title('per wall-clock')
ax.legend()
plt.tight_layout(); plt.show()


## The fitted functions

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
with torch.no_grad():
    ax.plot(x_plot, target(x_plot), 'k--', label='true sin(2πx)', lw=1.2, alpha=0.6)
    ax.scatter(x_train, y_train, s=18, color='#1B2D3D', zorder=5, label='data')
    ax.plot(x_plot, predict(sgd_theta, x_plot), color='#3D9B8F', lw=1.8, label='SGD fit')
    ax.plot(x_plot, predict(lbfgs_theta, x_plot), color='#D4A24C', lw=1.8, label='L-BFGS fit')
    ax.plot(x_plot, predict(gn_theta, x_plot), color='#B23A48', lw=1.8, label='Gauss-Newton fit')
ax.legend(); ax.grid(alpha=0.3); plt.show()


## The Jacobian ∂y/∂x: input sensitivity

A single backward pass per sample point gives the network's input sensitivity. For a net fitted to `sin(2πx)`, this should match the true derivative `2π cos(2πx)`.


In [ ]:
x_sweep = torch.linspace(0, 1, 200, dtype=torch.float64).unsqueeze(1).requires_grad_(True)
y_sweep = predict(gn_theta, x_sweep)
dydx = torch.autograd.grad(y_sweep.sum(), x_sweep)[0].detach().numpy().ravel()

fig, ax = plt.subplots(figsize=(9, 4))
xs = x_sweep.detach().numpy().ravel()
ax.plot(xs, dydx, color='#3D9B8F', lw=2, label='network ∂y/∂x')
ax.plot(xs, 2 * np.pi * np.cos(2 * np.pi * xs), 'k--', lw=1, alpha=0.6, label='true 2π cos(2πx)')
ax.axhline(0, color='gray', lw=0.5); ax.legend(); ax.grid(alpha=0.3); plt.show()


## The Hessian eigenvalue spectrum: why plain Newton fails

The loss is non-convex. At the initial weights, the Hessian has negative eigenvalues. Newton's direction `-H⁻¹g` points uphill along those directions, so plain Newton is attracted to saddle points (Dauphin et al. 2014). This is why we use Gauss-Newton instead — `J^T J` is positive semi-definite by construction.


In [ ]:
theta_init = init_params()
H_init = torch.autograd.functional.hessian(loss_from_vec, theta_init.clone().requires_grad_(True))
eig_init = torch.linalg.eigvalsh(0.5 * (H_init + H_init.T)).detach().numpy()
print(f'Negative eigenvalues at init: {(eig_init < 0).sum()} / {len(eig_init)}')

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#B23A48' if e < 0 else '#1B2D3D' for e in np.sort(eig_init)]
ax.bar(range(len(eig_init)), np.sort(eig_init), color=colors)
ax.axhline(0, color='gray'); ax.set_xlabel('index (sorted)'); ax.set_ylabel('Hessian eigenvalue')
ax.set_title('Hessian at initialisation — negative eigenvalues are saddle directions')
ax.grid(alpha=0.3); plt.show()


## Activation functions and their second derivatives

`tanh` has a smooth second derivative (Bishop's 1995 default). `ReLU` is piecewise linear, so its second derivative is 0 almost everywhere — Newton-style methods have no curvature to use. `GELU`, the activation in modern transformers, is smooth like `tanh` but doesn't saturate the way `tanh` does. We use `GELU` here for both reasons.


In [ ]:
def second_derivative(f, a):
    a2 = a.detach().clone().requires_grad_(True)
    y = f(a2)
    g = torch.autograd.grad(y.sum(), a2, create_graph=True)[0]
    g2 = torch.autograd.grad(g.sum(), a2)[0]
    return y.detach().numpy(), g.detach().numpy(), g2.detach().numpy()

a = torch.linspace(-4, 4, 400, dtype=torch.float64)
a_np = a.numpy()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True)
for ax, (name, f, col) in zip(axes, [('tanh', torch.tanh, '#3D9B8F'), ('ReLU', F.relu, '#B23A48'), ('GELU', F.gelu, '#D4A24C')]):
    y1, d1, d2 = second_derivative(f, a)
    ax.plot(a_np, y1, color=col, lw=1.8, label='h(a)')
    ax.plot(a_np, d1, color=col, lw=1.0, ls='--', label="h'(a)")
    ax.plot(a_np, d2, color='#1B2D3D', lw=1.4, ls=':', label="h''(a)")
    ax.axhline(0, color='gray', lw=0.5); ax.set_title(name); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Exercises

1. **Plain Newton**: replace `J^T J` in the Gauss-Newton trainer with the full Hessian (computed via `torch.autograd.functional.hessian`) and watch it get stuck around loss 0.27. Verify by inspecting the Hessian's eigenvalues during training.
2. **Swap GELU for ReLU**: change the activation. Train with all three methods. Explain why Gauss-Newton still works (hint: `J^T J` only needs the residual Jacobian, which exists for ReLU in a subgradient sense) but plain Newton gets worse.
3. **Scaling**: increase `HIDDEN` to 32, then 128. Plot the wall-clock of one Gauss-Newton iteration versus the wall-clock of 100 L-BFGS iterations as a function of `HIDDEN`. At what size does L-BFGS overtake Gauss-Newton?
4. **Hessian-vector products**: implement `Hv` using `torch.autograd.grad` twice without materialising `H`. Time it against `torch.autograd.functional.hessian(loss, theta) @ v`. You should see `O(W)` vs `O(W^2)`.
